# MLB Edge Finder — Exploration

Interactive walkthrough of the pipeline stages.

In [1]:
import logging
from datetime import date

from mlb_edge_finder import config

from pybaseball import cache
cache.enable()

config.setup_logging(level=logging.INFO)

## 1. Fetch Odds

In [2]:
from mlb_edge_finder import odds_ingestion

game_date = date.today()
odds_df = odds_ingestion.fetch_odds(game_date, force=True, debug=True)
odds_df.head()

2026-04-22 17:01:58,516 | INFO | mlb_edge_finder.odds_ingestion | Excluded 5 already-started game(s) — live in-game odds are not used
2026-04-22 17:01:58,519 | INFO | mlb_edge_finder.odds_ingestion | Raw API response: 12 game(s) returned
2026-04-22 17:01:58,520 | INFO | mlb_edge_finder.odds_ingestion |   game_id=2a8e5f1a3a4ac585a773ec30d535e4e0  home=Texas Rangers  away=Pittsburgh Pirates  commence_time=2026-04-23T00:06:00Z  local_date=2026-04-22
2026-04-22 17:01:58,520 | INFO | mlb_edge_finder.odds_ingestion |   game_id=863d10a0d1fd4ccb9810966d4740d766  home=Colorado Rockies  away=San Diego Padres  commence_time=2026-04-23T00:41:00Z  local_date=2026-04-22
2026-04-22 17:01:58,522 | INFO | mlb_edge_finder.odds_ingestion |   game_id=dfff7f7620a1af42549f691892c9c7cd  home=Arizona Diamondbacks  away=Chicago White Sox  commence_time=2026-04-23T01:41:00Z  local_date=2026-04-22
2026-04-22 17:01:58,522 | INFO | mlb_edge_finder.odds_ingestion |   game_id=a51b58a3d9fe032b72e59da66a7243fc  home=S

,game_id,home_team,away_team,home_odds_american,away_odds_american,commence_time
0,2a8e5f1a3a4ac585a773ec30d535e4e0,Texas Rangers,Pittsburgh Pirates,105,-110,2026-04-23T00:06:00Z
1,863d10a0d1fd4ccb9810966d4740d766,Colorado Rockies,San Diego Padres,140,-149,2026-04-23T00:41:00Z
2,a51b58a3d9fe032b72e59da66a7243fc,San Francisco Giants,Los Angeles Dodgers,176,-191,2026-04-23T01:46:00Z
3,dfff7f7620a1af42549f691892c9c7cd,Arizona Diamondbacks,Chicago White Sox,-145,133,2026-04-23T01:41:00Z


## 2. Fetch Stats

In [3]:
from mlb_edge_finder import stats_ingestion

stats_df = stats_ingestion.fetch_stats(date(2025, 4, 1), game_date)
stats_df.head()

2026-04-22 17:02:01,285 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs attempt 1/3 failed: Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Received status code 403 — retrying in 2s
2026-04-22 17:02:03,400 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs attempt 2/3 failed: Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Received status code 403 — retrying in 4s
2026-04-22 17:02:07,491 | WARNING | mlb_edge_finder.stats_ingestion | FanGraphs failed after 3 attempts (Error accessing 'https://www.fangraphs.com/leaders-legacy.aspx'. Received status code 403) — falling back to MLB Stats API
2026-04-22 17:02:07,983 | INFO | mlb_edge_finder.stats_ingestion | Wrote 30 rows to /Users/jaydengould/Documents/projects/mlb-edge-finder/data/raw/stats_2025-04-01.csv (source: mlb_api)


,team_abbr,bat_avg,obp,slg,ops,runs_per_game,era,whip,k_per_9,bb_per_9,fip_computed,data_source
0,TOR,.265,.333,.427,.760,4.925926,4.19,1.27,8.95,3.24,4.129138,mlb_api
1,PHI,.258,.328,.431,.759,4.802469,3.79,1.23,9.19,2.72,3.611079,mlb_api
2,MIL,.258,.332,.403,.735,4.975309,3.58,1.23,8.94,3.33,3.789390,mlb_api
3,BOS,.254,.324,.421,.745,4.851852,3.70,1.29,8.46,3.29,3.840560,mlb_api
4,ATH,.253,.318,.431,.749,4.524691,4.70,1.36,8.28,3.57,4.506805,mlb_api


## 3. Build Features

In [7]:
from mlb_edge_finder import features

features_df = features.build_features(game_date)
features_df.head()
features_df[features_df.isnull().any(axis=1)]
features_df[['home_team', 'away_team', 'home_bat_avg', 'away_bat_avg', 'home_era', 'away_era']]

2026-04-22 17:04:16,078 | INFO | mlb_edge_finder.features | Wrote 3 rows to /Users/jaydengould/Documents/projects/mlb-edge-finder/data/processed/features_2026-04-22.csv


,home_team,away_team,home_bat_avg,away_bat_avg,home_era,away_era
0,Texas Rangers,Pittsburgh Pirates,0.240,0.250,3.45,3.31
1,Colorado Rockies,San Diego Padres,0.238,0.230,4.26,3.22
2,San Francisco Giants,Los Angeles Dodgers,0.251,0.286,3.97,3.41


## 4a. Historical Ingestion

Fetch completed regular season results for each training season via `statsapi`.

In [ ]:
from mlb_edge_finder import historical_ingestion

# Fetch (or load cached) results for a single season
hist_2024 = historical_ingestion.fetch_historical(2024)
print(f"{len(hist_2024)} games")
hist_2024.head()

In [ ]:
# Concatenate all training seasons (2023, 2024, 2025)
all_hist = historical_ingestion.fetch_all_historical()
print(f"{len(all_hist)} total games")
all_hist.groupby(all_hist['game_date'].str[:4])['home_win'].agg(['count', 'mean'])

## 4b. Training Data

Join end-of-season team stats (one snapshot per year) to each game row to produce the model training set.

In [ ]:
from mlb_edge_finder import training_data

seasons = [2023, 2024, 2025]
training_df = training_data.build_training_set(seasons)
print(f"{len(training_df)} rows, {training_df.shape[1]} columns")
training_df.head()

In [ ]:
# Class balance and missing-value check
print("home_win distribution:")
print(training_df['home_win'].value_counts())
print()
nulls = training_df.isnull().sum()
print("Null counts:", nulls[nulls > 0].to_dict() or "none")

## 4c. Model Training

Train an XGBoost classifier on the training set, evaluate it, and persist the model and metrics.

In [ ]:
from mlb_edge_finder import model

clf = model.train(training_df)
clf

In [ ]:
# Evaluate on the held-out test split and save
# X_test and y_test come from the same 80/20 split inside train() — update if train() is
# changed to also return them.
# metrics = model.evaluate(clf, X_test, y_test)
# print(metrics)
# model.save_model(clf, metrics, date.today())

In [ ]:
# Reload a saved model from disk
# clf_loaded = model.load_model(date.today())
# clf_loaded

## 5. Find Edges

In [ ]:
from mlb_edge_finder import edge_finder

# edges = edge_finder.find_edges(features_df, clf)
# edges